In [ ]:
# src/kmeans_logic.py
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import PCA, VectorAssembler
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def run_kmeans_and_visualize(spark, df, k=6):
    """
    Thực hiện K-Means clustering và vẽ biểu đồ trực quan hóa.
    """
    # 1. Clustering
    kmeans = KMeans(featuresCol="features_scaled_final", k=k, seed=42)
    model = kmeans.fit(df)
    predictions = model.transform(df)

    # 2. Xuất bảng tâm cụm (Centroids) - Rất quan trọng cho báo cáo business
    centers = model.clusterCenters()
    # Giả định final_features là danh sách tên cột của bạn
    # Bạn có thể cần truyền danh sách tên cột vào hàm này
    print(f"\n--- TÂM CỤM (CENTROIDS) K={k} ---")

    # 3. PCA để trực quan hóa (k=2 để vẽ đồ thị 2D)
    pca = PCA(k=2, inputCol="features_scaled_final", outputCol="pca_2d")
    pca_model = pca.fit(predictions)
    predictions_2d = pca_model.transform(predictions)

    # 4. Trực quan hóa
    df_viz = predictions_2d.select("pca_2d", "prediction").sample(fraction=0.05, seed=42).toPandas()
    df_viz['PC1'] = df_viz['pca_2d'].apply(lambda x: x[0])
    df_viz['PC2'] = df_viz['pca_2d'].apply(lambda x: x[1])

    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df_viz, x='PC1', y='PC2', hue='prediction', palette='Set1', alpha=0.7, s=30)
    plt.title(f"Phân bổ cụm (K={k})", fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.savefig(f"Kmeans_K{k}_Viz.png", dpi=300)
    plt.show()

    return predictions

In [ ]:
# src/kmeans_logic.py (Tiếp tục)

def plot_cluster_distribution(predictions, output_path="Tỷ lệ_cluster.png"):
    """
    Hàm vẽ biểu đồ phân bổ thời gian/tỷ trọng các cluster.
    Input: predictions (Spark DataFrame)
    Output: File ảnh biểu đồ được lưu tại output_path
    """
    # Xử lý dữ liệu
    cluster_counts = predictions.groupBy("prediction").count().toPandas()
    total_points = cluster_counts['count'].sum()
    cluster_counts['Tỷ lệ (%)'] = (cluster_counts['count'] / total_points) * 100
    cluster_counts = cluster_counts.sort_values(by="prediction").reset_index(drop=True)
    cluster_counts.rename(columns={"prediction": "Cluster_ID", "count": "Số lượng (điểm dữ liệu)"}, inplace=True)

    # In bảng ra console để Master kiểm soát
    print("\n--- BẢNG PHÂN BỔ THỜI GIAN VẬN HÀNH ---")
    print(cluster_counts.round(2).to_string(index=False))

    # Vẽ biểu đồ
    plt.figure(figsize=(9, 9))
    explode = [0.05] * len(cluster_counts)
    plt.pie(
        cluster_counts['Tỷ lệ (%)'],
        labels=[f"Cluster {i}" for i in cluster_counts['Cluster_ID']],
        autopct='%1.2f%%',
        startangle=140,
        colors=sns.color_palette("Set1", n_colors=len(cluster_counts)),
        explode=explode,
        shadow=True
    )
    plt.title("Tỷ trọng thời gian phân bổ các trạng thái máy nén khí (K=6)", fontsize=15, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight', pad_inches=0.2)
    plt.show()
    print(f"Đã lưu biểu đồ tại: {output_path}")

In [ ]:
# src/kmeans_logic.py
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi

def plot_radar_chart(df_centroids, final_features, output_path="radar_chart_clusters.png"):
    """
    Hàm vẽ Radar Chart cho các tâm cụm (Centroids).
    df_centroids: Pandas DataFrame chứa dữ liệu tọa độ tâm cụm.
    final_features: Danh sách tên các đặc trưng (trục của biểu đồ).
    """
    N = len(final_features)
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1] # Đóng kín vòng tròn

    plt.figure(figsize=(12, 10))
    ax = plt.subplot(111, polar=True)

    # Thiết lập hướng của biểu đồ
    ax.set_theta_offset(pi / 2)
    ax.set_theta_direction(-1)
    plt.xticks(angles[:-1], final_features, size=11, fontweight='bold')

    # Thiết lập khoảng cách và giới hạn
    ax.set_rlabel_position(0)
    plt.yticks([-2, -1, 0, 1, 2, 4], ["-2", "-1", "0", "1", "2", "4"], color="grey", size=9)
    plt.ylim(-3, 6)

    # Vẽ từng cụm
    colors = sns.color_palette("Set1", n_colors=len(df_centroids))
    for i in range(len(df_centroids)):
        values = df_centroids.iloc[i].values.flatten().tolist()
        values += values[:1]
        ax.plot(angles, values, color=colors[i], linewidth=2.5, linestyle='solid', label=f"Cluster {i}")
        ax.fill(angles, values, color=colors[i], alpha=0.15)

    plt.title("Hồ sơ đặc trưng (Radar Chart) của các trạng thái máy nén khí", size=16, fontweight='bold', y=1.08)
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), title="Trạng thái (Cluster)", fontsize=10)
    plt.grid(color='#AAAAAA', linestyle='--', linewidth=0.5)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight', pad_inches=0.2)
    plt.show()
    print(f"Đã lưu Radar Chart tại: {output_path}")